# Feature Engineering — Target Encoding (Store-Family)

| Feature | Mô tả |
|---|---|
| `store_family_te` | Mean sales theo cặp store-family, tính bằng KFold OOF (5 folds, smoothing=10) |

**Anti-leakage**: Out-of-fold — mỗi observation không tham gia vào mean của chính nó.  
**Test set**: Encode bằng encoder fit trên toàn bộ train (an toàn, test không có target).

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from category_encoders import TargetEncoder

In [ ]:
import sys
from pathlib import Path

def _find_root():
    """Tìm project root chứa config.py."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'config.py').exists():
            return p
    raise RuntimeError("Không tìm thấy project root. Mở Jupyter từ thư mục gốc của project.")

_root = _find_root()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from config import PROCESSED_DIR

df_train = pd.read_csv(PROCESSED_DIR / 'train_cleaned.csv')
df_test  = pd.read_csv(PROCESSED_DIR / 'test_cleaned.csv')

print(f'Train shape : {df_train.shape}')
print(f'Test shape  : {df_test.shape}')

In [3]:
# Local aliases used by the explanatory cells below
# These point to the loaded dataframes so cells that reference
# `train` / `test` / `target` work when running the notebook.
train = df_train.copy()
test = df_test.copy()
target = 'sales'
# Ensure composite key exists so explanatory cells can run independently
train['store_family'] = train['store_nbr'].astype(str) + '_' + train['family']
test['store_family']  = test['store_nbr'].astype(str)  + '_' + test['family']
print('Aliases set: train/test/target and store_family created')

Aliases set: train/test/target and store_family created


---
## 1. Tại sao dùng KFold OOF Target Encoding?

Tính `mean(sales)` trực tiếp trên toàn dataset → mỗi observation đóng góp vào mean của chính nó → **data leakage**.

**KFold OOF**: với mỗi fold, encoder fit trên K-1 folds, transform fold còn lại — observation không bao giờ thấy target của chính mình. `smoothing=10` blend groups nhỏ về global mean.

---
## 2. Hàm `add_target_encoding`

In [4]:
def add_target_encoding(train, test, target="sales", n_splits=5):

    train, test = train.copy(), test.copy()

    # Combine key
    train["store_family"] = (
        train["store_nbr"].astype(str) + "_" + train["family"]
    )
    test["store_family"] = (
        test["store_nbr"].astype(str) + "_" + test["family"]
    )

    kf = KFold(n_splits=n_splits, shuffle=False)
    oof = np.zeros(len(train))

    # ---- Out-of-fold encoding (leakage safe) ----
    for tr_idx, val_idx in kf.split(train):

        enc = TargetEncoder(cols=["store_family"], smoothing=10)

        # Fit ONLY on training fold
        enc.fit(
            train.iloc[tr_idx][["store_family"]],
            train.iloc[tr_idx][target]
        )

        # Transform validation fold (never sees its own target)
        oof[val_idx] = enc.transform(
            train.iloc[val_idx][["store_family"]]
        )["store_family"]

    train["store_family_te"] = oof

    # ---- Fit on full train for test ----
    final_enc = TargetEncoder(cols=["store_family"], smoothing=10)
    final_enc.fit(train[["store_family"]], train[target])

    # Test uses only train statistics
    test["store_family_te"] = final_enc.transform(
        test[["store_family"]]
    )["store_family"]

    return train, test

### Bước 1 — Tạo composite key `store_family`

Ghép `store_nbr` và `family` thành một chuỗi duy nhất (ví dụ: `"1_AUTOMOTIVE"`).

**Về leakage**: Bước này **không liên quan** đến target — chỉ tạo identifier từ các feature có sẵn, nên hoàn toàn an toàn.

**Tại sao cần composite key?** Vì ta muốn tính mean sales cho **từng cặp** store-family, không phải chỉ theo store hoặc chỉ theo family. Mỗi cửa hàng bán các mặt hàng khác nhau với mức doanh số khác nhau.

In [5]:
train["store_family"] = train["store_nbr"].astype(str) + "_" + train["family"]

### Bước 2 — KFold Out-of-Fold Encoding (core leakage protection)

Đây là bước **quan trọng nhất** để chống data leakage.

**Tại sao chống được leakage?**

1. **`enc.fit()` chỉ nhận `tr_idx`** — encoder tính mean sales **chỉ từ 4/5 dữ liệu** (training fold). Các observation trong `val_idx` **không tham gia** vào phép tính này.

2. **`enc.transform()` chỉ áp dụng cho `val_idx`** — mỗi observation nhận giá trị mean được tính từ dữ liệu **không chứa chính nó**.

3. **Encoder mới cho mỗi iteration** — `enc = TargetEncoder(...)` được khởi tạo lại trong mỗi vòng lặp, đảm bảo không có thông tin "rò rỉ" giữa các fold.

4. **`smoothing=10`** — Tham số smoothing pha trộn mean của nhóm nhỏ với global mean, giảm overfitting cho các cặp store-family có ít observation. Công thức: `encoded = (count * group_mean + smoothing * global_mean) / (count + smoothing)`.

5. **`shuffle=False`** — Giữ thứ tự thời gian nguyên vẹn, phù hợp với bài toán time series (tránh trộn lẫn future data vào past folds).

In [6]:
kf = KFold(n_splits=5, shuffle=False)
oof = np.zeros(len(train))

for tr_idx, val_idx in kf.split(train):
    enc = TargetEncoder(cols=["store_family"], smoothing=10)
    enc.fit(train.iloc[tr_idx][["store_family"]], train.iloc[tr_idx][target])
    oof[val_idx] = enc.transform(train.iloc[val_idx][["store_family"]])["store_family"]

In [7]:
final_enc = TargetEncoder(cols=["store_family"], smoothing=10)
final_enc.fit(train[["store_family"]], train[target])
test["store_family_te"] = final_enc.transform(test[["store_family"]])["store_family"]

---
## 3. Feature Name Registry

In [8]:
TARGET_ENCODING_FEATURES = [
    'store_family_te',
]

print(f'Target encoding features ({len(TARGET_ENCODING_FEATURES)}):')
for f in TARGET_ENCODING_FEATURES:
    print(f'  - {f}')

Target encoding features (1):
  - store_family_te


---
## 4. Chạy Pipeline

In [9]:
df_train_encoded, df_test_encoded = add_target_encoding(df_train, df_test)

print('=== Train — Target Encoding Feature ===')
print(df_train_encoded['store_family_te'].describe())
print()
print('=== Test — Target Encoding Feature ===')
print(df_test_encoded['store_family_te'].describe())
print()
print('=== Sample rows (Train) ===')
df_train_encoded[['store_nbr', 'family', 'store_family', 'sales', 'store_family_te']].head(10)

=== Train — Target Encoding Feature ===
count    3.000888e+06
mean     3.577729e+02
std      9.649663e+02
min      0.000000e+00
25%      2.691908e+00
50%      1.942168e+01
75%      2.225125e+02
max      1.025046e+04
Name: store_family_te, dtype: float64

=== Test — Target Encoding Feature ===
count    28512.000000
mean       357.775749
std        961.600889
min          0.000000
25%          2.877078
50%         19.812350
75%        221.086105
max       9730.436483
Name: store_family_te, dtype: float64

=== Sample rows (Train) ===


,store_nbr,family,store_family,sales,store_family_te
0,1,AUTOMOTIVE,1_AUTOMOTIVE,0.0,3.523385
1,1,BABY CARE,1_BABY CARE,0.0,0.000000
2,1,BEAUTY,1_BEAUTY,0.0,2.561247
3,1,BEVERAGES,1_BEVERAGES,0.0,1772.723088
4,1,BOOKS,1_BOOKS,0.0,0.156644
5,1,BREAD/BAKERY,1_BREAD/BAKERY,0.0,355.752459
6,1,CELEBRATION,1_CELEBRATION,0.0,12.461767
7,1,CLEANING,1_CLEANING,0.0,650.435783
8,1,DAIRY,1_DAIRY,0.0,680.178916
9,1,DELI,1_DELI,0.0,126.689168


In [ ]:
# CSV write disabled — target encoding được tính inline trong build_train_final.py.
# Để tạo CSV cho model training:
#   python scripts/build_train_final.py   →  data/processed/train_final.csv
#   python scripts/create_splits.py       →  data/processed/splits/
print('[SKIPPED] CSV write disabled')
print(f'Train encoded shape : {df_train_encoded.shape}')
print(f'Test encoded shape  : {df_test_encoded.shape}')